In [1]:
pip install -r requirements_dev.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install smartapi-python pyotp pandas openpyxl

In [3]:
pip install pyotp

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install logzero

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install websocket-client

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip uninstall pycrypto

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install pycryptodome

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install SmartApi

  Using cached SmartAPI-1.1.0-py3-none-any.whl
  Using cached pycrypto-2.6.1.tar.gz (446 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build PyCrypto
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for PyCrypto (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      C:\Users\Kuro Gaming\AppData\Local\Temp\pip-build-env-q9tnn3a7\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: Public Domain
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudi

In [9]:
!pip install watchdog

In [13]:
print("🚀 DAY 15: PORTFOLIO RISK MANAGEMENT")
print("NIFTY → NIFTY50 universe + Correlation VaR")

import pandas as pd
import numpy as np
from SmartApi.smartConnect import SmartConnect
import pyotp
import time

# 1. ANGEL ONE CONNECTION (Your validated session)
API_KEY = "hYMUaSKT"
CLIENT_CODE = "AAAQ167458"
angel = SmartConnect(api_key=API_KEY)
session = angel.generateSession(CLIENT_CODE, "9161", pyotp.TOTP("IIBHSWOSJYPGFBWM75R6NHSWTI").now())

# 2. NIFTY50 TOP TOKENS (use your actual instrument‑list tokens)
#   Example: on Angel SmartAPI, Nifty index is often 99926000, not 26000.
#   Replace with your real token file or script.
NIFTY50 = {
    "NIFTY": "26000",      # Nifty index (check with Angel script list)
    "RELIANCE": "2885",       # RELIANCE EQ (verify in your token list)
    "TCS": "11536",            # TCS EQ (verify in your token list)
    "HDFCBANK": "1333",       # HDFCBANK EQ (verify in your token list)
    "INFY": "1594"            # INFY EQ (verify in your token list)
}

# 3. MULTI-ASSET EVENT STORE
class MultiAssetEventStore:
    def __init__(self):
        self.assets = {}
    
    def collect_live_data(self, symbols, count=20):
        """Live NSE data collection (finite execution)"""
        print(f"📡 Collecting {count} ticks x {len(symbols)} assets...")
        
        for symbol in symbols:
            token = NIFTY50[symbol]
            self.assets[symbol] = []
            
            for i in range(count):
                ltp_data = angel.ltpData("NSE", symbol, token)
                if ltp_data and ltp_data.get("status") and ltp_data.get("data"):
                    ltp = ltp_data["data"]["ltp"]
                    self.assets[symbol].append({
                        "ltp": ltp,
                        "time": pd.Timestamp.now()
                    })
                time.sleep(0.5)  # Safe rate limit
            
            if self.assets[symbol]:
                last_ltp = self.assets[symbol][-1]["ltp"]
                print(f"✅ {symbol}: {len(self.assets[symbol])} ticks @ ₹{last_ltp:.0f}")
            else:
                print(f"❌ {symbol}: 0 ticks (check symbol/token)")

    def compute_returns(self):
        """Portfolio returns matrix (defensive)"""
        data = []
        for symbol, events in self.assets.items():
            if not events:
                print(f"⚠️ Skipping {symbol}: no ticks")
                continue

            df = pd.DataFrame(events)

            if "ltp" not in df.columns:
                print(f"⚠️ Skipping {symbol}: no 'ltp' column")
                continue

            df["symbol"] = symbol
            df["returns"] = df["ltp"].pct_change()
            data.append(df)
        
        if data:
            self.returns_df = pd.concat(data, ignore_index=True)
        else:
            self.returns_df = pd.DataFrame()
        
        return self.returns_df


# 4. INSTITUTIONAL PORTFOLIO RISK ENGINE
class PortfolioRiskEngine:
    def __init__(self, event_store):
        self.event_store = event_store
        self.weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])  # Equal weight (5 assets)

    def portfolio_var(self):
        """Correlation-adjusted portfolio VaR 99%"""
        if len(self.event_store.returns_df) == 0:
            print("⚠️ No returns data; cannot compute VaR.")
            return {}

        # Pivot: time × symbol, drop times where all are missing
        returns_pivot = self.event_store.returns_df \
            .pivot(index="time", columns="symbol", values="returns") \
            .dropna(how="all")

        if returns_pivot.shape[0] < 2:
            print("⚠️ Insufficient return observations.")
            return {}

        returns_matrix = returns_pivot.fillna(0).values
        cov_matrix = np.cov(returns_matrix.T)
        portfolio_vol = np.sqrt(self.weights @ cov_matrix @ self.weights.T)
        var_99 = 2.33 * portfolio_vol  # 99% confidence (assuming normal)

        print(f"\n📊 PORTFOLIO RISK METRICS:")
        print(f"• Portfolio Vol:   {portfolio_vol:.4f}")
        print(f"• VaR 99%:         {var_99:.4f}")
        print(f"• ₹5L Exposure:    ₹{abs(var_99 * 500000):,.0f}")
        
        return {"portfolio_vol": portfolio_vol, "var_99": var_99}


# 5. EXECUTE DAY 15
print("\n🎯 DAY 15 LIVE EXECUTION START...")
event_store = MultiAssetEventStore()
event_store.collect_live_data(["NIFTY", "RELIANCE", "TCS","HDFCBANK","INFY"], count=10)  # 3 assets first
event_store.compute_returns()

risk_engine = PortfolioRiskEngine(event_store)
portfolio_risk = risk_engine.portfolio_var()

print(f"\n✅ DAY 15 COMPLETE: NIFTY50 Risk Engine")
print(f"📦 Assets: {len(event_store.assets)} | Ticks: {sum(len(e) for e in event_store.assets.values())}")
print("🎯 GitHub: Day15_MultiAssetRisk.ipynb")


🚀 DAY 15: PORTFOLIO RISK MANAGEMENT
NIFTY → NIFTY50 universe + Correlation VaR


[I 260323 10:20:26 smartConnect:121] in pool



🎯 DAY 15 LIVE EXECUTION START...
📡 Collecting 10 ticks x 5 assets...
✅ NIFTY: 10 ticks @ ₹22551
✅ RELIANCE: 10 ticks @ ₹1396
✅ TCS: 10 ticks @ ₹2401
✅ HDFCBANK: 10 ticks @ ₹752
✅ INFY: 10 ticks @ ₹1248

📊 PORTFOLIO RISK METRICS:
• Portfolio Vol:   0.0000
• VaR 99%:         0.0001
• ₹5L Exposure:    ₹30

✅ DAY 15 COMPLETE: NIFTY50 Risk Engine
📦 Assets: 5 | Ticks: 50
🎯 GitHub: Day15_MultiAssetRisk.ipynb


In [ ]:
#Var is near 0, To avoid this, Perplexity suggested to day data

In [14]:
print("🚀 DAY 15: PORTFOLIO RISK MANAGEMENT")
print("NIFTY → NIFTY50 universe + Correlation VaR")

import pandas as pd
import numpy as np
from SmartApi.smartConnect import SmartConnect
import pyotp
import time

# 1. ANGEL ONE CONNECTION (Your validated session)
API_KEY = "hYMUaSKT"
CLIENT_CODE = "AAAQ167458"
angel = SmartConnect(api_key=API_KEY)
session = angel.generateSession(CLIENT_CODE, "9161", pyotp.TOTP("IIBHSWOSJYPGFBWM75R6NHSWTI").now())

# 2. NIFTY50 TOKEN MAP (use your instrument‑list tokens)
#   Replace with your real token file if needed.
NIFTY50 = {
    "NIFTY": "26000",      # Nifty index (check your script list)
    "RELIANCE": "2885",       # RELIANCE EQ
    "TCS": "3045",            # TCS EQ
    "HDFCBANK": "1333",       # HDFCBANK EQ
    "INFY": "1594"            # INFY EQ
}

# 3. MULTI‑ASSET EVENT STORE
class MultiAssetEventStore:
    def __init__(self):
        self.assets = {}

    def collect_live_data(self, symbols, count=20):
        """Live NSE data collection (finite execution)"""
        print(f"📡 Collecting {count} ticks x {len(symbols)} assets...")
        for symbol in symbols:
            token = NIFTY50[symbol]
            self.assets[symbol] = []

            for i in range(count):
                ltp_data = angel.ltpData("NSE", symbol, token)
                if ltp_data and ltp_data.get("status") and ltp_data.get("data"):
                    ltp = ltp_data["data"]["ltp"]
                    self.assets[symbol].append({
                        "ltp": ltp,
                        "time": pd.Timestamp.now()
                    })
                time.sleep(0.5)  # Safe rate limit

            if self.assets[symbol]:
                last_ltp = self.assets[symbol][-1]["ltp"]
                print(f"✅ {symbol}: {len(self.assets[symbol])} ticks @ ₹{last_ltp:.0f}")
            else:
                print(f"❌ {symbol}: 0 ticks (check symbol/token)")

    def compute_returns(self):
        """Portfolio returns matrix (defensive)"""
        data = []
        for symbol, events in self.assets.items():
            if not events:
                print(f"⚠️ Skipping {symbol}: no ticks")
                continue

            df = pd.DataFrame(events)
            if "ltp" not in df.columns:
                print(f"⚠️ Skipping {symbol}: no 'ltp' column")
                continue

            df["symbol"] = symbol
            df["returns"] = df["ltp"].pct_change()
            data.append(df)

        if data:
            self.returns_df = pd.concat(data, ignore_index=True)
        else:
            self.returns_df = pd.DataFrame()

        return self.returns_df


# 4. INSTITUTIONAL PORTFOLIO RISK ENGINE (with meaningful VaR)
class PortfolioRiskEngine:
    def __init__(self, event_store):
        self.event_store = event_store
        self.weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])  # equal weights for 5 assets

    def portfolio_var(self):
        """Correlation‑adjusted portfolio VaR 99% (1‑day‑equivalent from ticks)"""
        if len(self.event_store.returns_df) == 0:
            print("⚠️ No returns data; cannot compute VaR.")
            return {}

        # Pivot: time × symbol; drop times where all are missing
        returns_pivot = self.event_store.returns_df \
            .pivot(index="time", columns="symbol", values="returns") \
            .dropna(how="all")

        if returns_pivot.shape[0] < 2:
            print("⚠️ Insufficient return observations.")
            return {}

        # Resize weights to actual number of symbols
        n_assets = returns_pivot.shape[1]
        if n_assets != len(self.weights):
            print(f"⚠️ weights ({len(self.weights)}) ≠ symbols ({n_assets}); adjusting to equal weights.")
            weights = np.ones(n_assets) / n_assets
        else:
            weights = self.weights

        returns_matrix = returns_pivot.fillna(0).values
        cov_matrix = np.cov(returns_matrix.T)
        portfolio_vol_tick = np.sqrt(weights @ cov_matrix @ weights.T)

        # --- Scale to 1‑day VaR from tick‑level volatility ---
        # 1 tick = 0.5–1 sec, 1 trading day = 7.5 hrs ≈ 27,000 seconds
        # Assume your count is over ~10–20 minutes of data:
        #  10 minutes ≈ 600 seconds → scale ≈ sqrt(27000 / 600) ≈ 6.7
        # Adjust HORIZON_SCALE to your tick‑sampling window and holding horizon.
        HORIZON_SCALE = 7.0   # approximate tick‑to‑day scaling factor (configurable)

        portfolio_vol_day = portfolio_vol_tick * HORIZON_SCALE
        var_99 = 2.33 * portfolio_vol_day  # 99% 1‑day VaR (normal)

        print(f"\n📊 PORTFOLIO RISK METRICS (1‑day equivalent):")
        print(f"• Tick‑level Vol:  {portfolio_vol_tick:.6f}")
        print(f"• 1‑day Vol:       {portfolio_vol_day:.4f}")
        print(f"• VaR 99% (1‑day): {var_99:.4f}")
        print(f"• ₹5L Exposure (1‑day): ₹{abs(var_99 * 500000):,.0f}")

        return {
            "portfolio_vol_tick": portfolio_vol_tick,
            "portfolio_vol_day": portfolio_vol_day,
            "var_99": var_99
        }


# 5. EXECUTE DAY 15 (meaningful VaR)
print("\n🎯 DAY 15 LIVE EXECUTION START...")
event_store = MultiAssetEventStore()
event_store.collect_live_data(
    ["NIFTY", "RELIANCE", "TCS", "HDFCBANK", "INFY"],
    count=20   # 20 ticks ≈ 10–20 seconds per asset
)
event_store.compute_returns()

risk_engine = PortfolioRiskEngine(event_store)
portfolio_risk = risk_engine.portfolio_var()

print(f"\n✅ DAY 15 COMPLETE: NIFTY50 Risk Engine")
print(f"📦 Assets: {len(event_store.assets)} | Ticks: {sum(len(e) for e in event_store.assets.values())}")
print("🎯 GitHub: Day15_MultiAssetRisk.ipynb")


🚀 DAY 15: PORTFOLIO RISK MANAGEMENT
NIFTY → NIFTY50 universe + Correlation VaR


[I 260323 10:24:01 smartConnect:121] in pool



🎯 DAY 15 LIVE EXECUTION START...
📡 Collecting 20 ticks x 5 assets...
✅ NIFTY: 20 ticks @ ₹22554
✅ RELIANCE: 20 ticks @ ₹1392
✅ TCS: 20 ticks @ ₹1017
✅ HDFCBANK: 20 ticks @ ₹752
✅ INFY: 20 ticks @ ₹1249

📊 PORTFOLIO RISK METRICS (1‑day equivalent):
• Tick‑level Vol:  0.000033
• 1‑day Vol:       0.0002
• VaR 99% (1‑day): 0.0005
• ₹5L Exposure (1‑day): ₹269

✅ DAY 15 COMPLETE: NIFTY50 Risk Engine
📦 Assets: 5 | Ticks: 100
🎯 GitHub: Day15_MultiAssetRisk.ipynb
